# Graphen-Drill: f ↔ f' ↔ f'' ↔ F zuordnen

Übung macht den Meister! Hier kannst du das **Zuordnen von Graphen** trainieren — die wichtigste Fähigkeit für den hilfsmittelfreien Teil A.

**So funktioniert's:**
1. Wähle eine Schwierigkeitsstufe
2. Dir werden mehrere Graphen angezeigt — ordne sie zu
3. Klicke auf "Lösung", um dein Ergebnis zu prüfen
4. Klicke auf "Neue Aufgabe" für die nächste Runde

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from ipywidgets import interact, Dropdown, Button, Output, VBox, HBox, Label
from IPython.display import display, Markdown, clear_output
import random

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 13
plt.rcParams['mathtext.fontset'] = 'cm'

x_sym = sp.Symbol('x')

# ── Funktionspool ──────────────────────────────────────────────
# Jeder Eintrag: (sympy-Ausdruck, x-Bereich, Name)
POOL = [
    # Polynome
    (x_sym**3 - 3*x_sym, (-2.5, 2.5)),
    (x_sym**4 - 4*x_sym**2 + 1, (-2.5, 2.5)),
    (-x_sym**3 + 6*x_sym**2 - 9*x_sym, (-0.5, 4.5)),
    (x_sym**3 - 6*x_sym**2 + 9*x_sym + 1, (-0.5, 4.5)),
    (x_sym**4 - 2*x_sym**2, (-2, 2)),
    (-x_sym**4 + 8*x_sym**2 - 7, (-2.5, 2.5)),
    (x_sym**2 * (x_sym - 2), (-1, 3)),
    # e-Funktionen
    (x_sym * sp.exp(-x_sym), (-1, 5)),
    ((x_sym**2 - 1) * sp.exp(-x_sym), (-1, 5)),
    (sp.exp(-x_sym**2 / 2), (-3, 3)),
    ((2*x_sym - x_sym**2) * sp.exp(x_sym), (-4, 1.5)),
    # Trigonometrisch
    (sp.sin(x_sym), (-2*float(sp.pi), 2*float(sp.pi))),
    (sp.sin(x_sym) + sp.Rational(1,2)*sp.sin(2*x_sym), (-2*float(sp.pi), 2*float(sp.pi))),
]

FARBEN = ['#58C4DD', '#FC6255', '#FF8C00', '#50C878']
GRAPH_NAMEN = ['Graph A', 'Graph B', 'Graph C', 'Graph D']

def erzeuge_aufgabe(stufe):
    """Erzeugt eine zufällige Zuordnungsaufgabe."""
    f_sym, (xmin, xmax) = random.choice(POOL)
    fp_sym = sp.diff(f_sym, x_sym)
    fpp_sym = sp.diff(fp_sym, x_sym)
    F_sym = sp.integrate(f_sym, x_sym)

    if stufe == 1:
        graphen = [('f', f_sym), ("f'", fp_sym)]
    elif stufe == 2:
        graphen = [('f', f_sym), ("f'", fp_sym), ("f''", fpp_sym)]
    else:
        graphen = [('f', f_sym), ("f'", fp_sym), ("f''", fpp_sym), ('F', F_sym)]

    # Zufällige Reihenfolge
    random.shuffle(graphen)
    return f_sym, graphen, (xmin, xmax)

def plot_aufgabe(graphen, xlim, ax_list):
    """Plottet die Graphen in zufälliger Reihenfolge."""
    xs = np.linspace(xlim[0], xlim[1], 500)
    for ax, (label, sym_expr), farbe, name in zip(ax_list, graphen, FARBEN, GRAPH_NAMEN):
        f_num = sp.lambdify(x_sym, sym_expr, 'numpy')
        ys = f_num(xs)
        # y-Bereich begrenzen für Lesbarkeit
        median = np.median(np.abs(ys[np.isfinite(ys)]))
        clip = max(10, 5 * median)
        ys = np.clip(ys, -clip, clip)
        ax.plot(xs, ys, color=farbe, linewidth=2.5)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.axvline(x=0, color='k', linewidth=0.5)
        ax.grid(True, alpha=0.3)
        ax.set_title(name, fontsize=14, fontweight='bold', color=farbe)
        ax.set_xlim(xlim)

print("✓ Setup geladen")

---
## Stufe 1: f und f' zuordnen (Einstieg)

Zwei Graphen werden angezeigt — welcher ist f, welcher ist f'?

**Tipps:**
- Wo f' = 0 ist (Nullstelle), hat f ein Extremum
- Wo f' > 0, steigt f
- f' hat immer einen Grad weniger als f

In [ ]:
out_stufe1 = Output()

def stufe1_neu(_=None):
    with out_stufe1:
        clear_output(wait=True)
        f_sym, graphen, xlim = erzeuge_aufgabe(stufe=1)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        plot_aufgabe(graphen, xlim, axes)
        plt.suptitle("Welcher Graph ist f, welcher ist f' ?", fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # Lösung
        loesung = f"**Funktion:** $f(x) = {sp.latex(f_sym)}$\n\n"
        for i, (label, sym_expr) in enumerate(graphen):
            loesung += f"- **{GRAPH_NAMEN[i]}** = ${label}(x) = {sp.latex(sym_expr)}$\n"

        # Erklärung generieren
        fp_sym = sp.diff(f_sym, x_sym)
        nullstellen_fp = [n for n in sp.solve(fp_sym, x_sym) if n.is_real]
        if nullstellen_fp:
            erkl = "\n\n**So erkennt man es:** "
            for ns in nullstellen_fp:
                erkl += f"$f'$ hat eine Nullstelle bei $x = {sp.latex(ns)}$, "
                fpp_val = sp.diff(fp_sym, x_sym).subs(x_sym, ns)
                if fpp_val < 0:
                    erkl += f"also hat $f$ dort ein **Maximum**. "
                elif fpp_val > 0:
                    erkl += f"also hat $f$ dort ein **Minimum**. "
                else:
                    erkl += f"also hat $f$ dort einen **Sattelpunkt**. "
            loesung += erkl

        display(Markdown(f'<details>\n<summary><b>Lösung aufklappen</b></summary>\n\n{loesung}\n\n</details>'))

btn1 = Button(description='Neue Aufgabe', button_style='info', icon='refresh')
btn1.on_click(stufe1_neu)
display(btn1)
display(out_stufe1)
stufe1_neu()

---
## Stufe 2: f, f' und f'' zuordnen

Drei Graphen — welcher gehört zu f, f' und f''?

**Zusätzliche Tipps:**
- f'' ist die Ableitung von f' (gleiche Logik nochmal anwenden!)
- Wo f'' = 0 mit VZW → f hat Wendepunkt
- Wo f'' > 0 → f ist linksgekrümmt (Mulde)

In [ ]:
out_stufe2 = Output()

def stufe2_neu(_=None):
    with out_stufe2:
        clear_output(wait=True)
        f_sym, graphen, xlim = erzeuge_aufgabe(stufe=2)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        plot_aufgabe(graphen, xlim, axes)
        plt.suptitle("Welcher Graph ist f, f' und f'' ?", fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.show()

        loesung = f"**Funktion:** $f(x) = {sp.latex(f_sym)}$\n\n"
        for i, (label, sym_expr) in enumerate(graphen):
            loesung += f"- **{GRAPH_NAMEN[i]}** = ${label}(x) = {sp.latex(sym_expr)}$\n"

        # Erklärungskette
        fp_sym = sp.diff(f_sym, x_sym)
        fpp_sym = sp.diff(fp_sym, x_sym)
        erkl = "\n\n**Erkennungsweg:**\n"
        erkl += "1. Den Graphen mit dem **höchsten Polynomgrad** (meiste Nullstellen/Extrema) finden → das ist f\n"
        erkl += "2. Prüfen: Nullstellen von f' müssen zu Extrema von f passen\n"
        erkl += "3. f'' hat den niedrigsten Grad und die wenigsten Besonderheiten\n"
        loesung += erkl

        display(Markdown(f'<details>\n<summary><b>Lösung aufklappen</b></summary>\n\n{loesung}\n\n</details>'))

btn2 = Button(description='Neue Aufgabe', button_style='info', icon='refresh')
btn2.on_click(stufe2_neu)
display(btn2)
display(out_stufe2)
stufe2_neu()

---
## Stufe 3: f, f', f'' und F zuordnen (Königsdisziplin)

Vier Graphen — welcher gehört zu f, f', f'' und der Stammfunktion F?

**Zusätzlicher Tipp für F:**
- F hat einen Grad MEHR als f
- Wo f > 0 → F steigt. Wo f < 0 → F fällt.
- Nullstellen von f → Extrema von F

In [ ]:
out_stufe3 = Output()

def stufe3_neu(_=None):
    with out_stufe3:
        clear_output(wait=True)
        f_sym, graphen, xlim = erzeuge_aufgabe(stufe=3)
        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        plot_aufgabe(graphen, xlim, axes)
        plt.suptitle("Welcher Graph ist F, f, f' und f'' ?", fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.show()

        loesung = f"**Funktion:** $f(x) = {sp.latex(f_sym)}$\n\n"
        for i, (label, sym_expr) in enumerate(graphen):
            loesung += f"- **{GRAPH_NAMEN[i]}** = ${label}(x) = {sp.latex(sym_expr)}$\n"

        erkl = "\n\n**Erkennungsweg:**\n"
        erkl += "1. **F** hat den höchsten Grad (meiste Wendungen) und die meisten Extrema\n"
        erkl += "2. **f** ist eine Stufe darunter — Nullstellen von f = Extrema von F\n"
        erkl += "3. **f'** — Nullstellen von f' = Extrema von f\n"
        erkl += "4. **f''** hat den niedrigsten Grad\n"
        loesung += erkl

        display(Markdown(f'<details>\n<summary><b>Lösung aufklappen</b></summary>\n\n{loesung}\n\n</details>'))

btn3 = Button(description='Neue Aufgabe', button_style='info', icon='refresh')
btn3.on_click(stufe3_neu)
display(btn3)
display(out_stufe3)
stufe3_neu()

---
## Spezialrunde: Aus f' die Eigenschaften von f ablesen

Hier siehst du nur den Graph von **f'(x)**. Beantworte die Fragen zu f, ohne f zu sehen!

In [ ]:
out_spezial = Output()

def spezial_neu(_=None):
    with out_spezial:
        clear_output(wait=True)
        f_sym, _, xlim = erzeuge_aufgabe(stufe=1)
        fp_sym = sp.diff(f_sym, x_sym)
        fpp_sym = sp.diff(fp_sym, x_sym)

        xs = np.linspace(xlim[0], xlim[1], 500)
        fp_num = sp.lambdify(x_sym, fp_sym, 'numpy')
        fp_vals = fp_num(xs)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(xs, fp_vals, color='#FC6255', linewidth=2.5, label=rf"$f'(x) = {sp.latex(fp_sym)}$")
        ax.axhline(y=0, color='k', linewidth=0.8)
        ax.axvline(x=0, color='k', linewidth=0.5)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(xlim)
        ax.legend(fontsize=13, loc='upper right', framealpha=0.9)
        ax.set_title("Graph von f'(x) — Was kannst du über f sagen?", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # Fragen
        display(Markdown("""
**Beantworte diese Fragen, indem du nur den Graph von f' betrachtest:**

1. In welchen Bereichen ist f **monoton steigend**? (Wo ist f' > 0?)
2. In welchen Bereichen ist f **monoton fallend**? (Wo ist f' < 0?)
3. Wo hat f **lokale Extrema**? Welche Art (Maximum/Minimum)?
4. Wo hat f **Wendepunkte**? (Wo hat f' Extrema?)
"""))

        # Lösung generieren
        nullstellen_fp = sorted([n for n in sp.solve(fp_sym, x_sym) if n.is_real], key=float)
        extrema_fp = sorted([n for n in sp.solve(fpp_sym, x_sym) if n.is_real], key=float)

        loesung = f"**Ausgangsfunktion:** $f(x) = {sp.latex(f_sym)}$\n\n"

        # Monotonie
        loesung += "**1./2. Monotonie:**\n"
        for i in range(len(nullstellen_fp)):
            ns = nullstellen_fp[i]
            # Vorzeichen links der Nullstelle
            if i == 0:
                test_x = float(ns) - 1
            else:
                test_x = (float(nullstellen_fp[i-1]) + float(ns)) / 2
            wert = fp_sym.subs(x_sym, sp.Rational(test_x).limit_denominator(100))
            if wert > 0:
                loesung += f"- f steigt links von $x = {sp.latex(ns)}$\n"
            else:
                loesung += f"- f fällt links von $x = {sp.latex(ns)}$\n"
        loesung += "\n"

        # Extrema
        loesung += "**3. Extrema von f** (Nullstellen von f' mit VZW):\n"
        for ns in nullstellen_fp:
            fpp_val = fpp_sym.subs(x_sym, ns)
            f_val = f_sym.subs(x_sym, ns)
            if fpp_val < 0:
                loesung += f"- $x = {sp.latex(ns)}$: **Hochpunkt** bei $({sp.latex(ns)} | {sp.latex(f_val)})$\n"
            elif fpp_val > 0:
                loesung += f"- $x = {sp.latex(ns)}$: **Tiefpunkt** bei $({sp.latex(ns)} | {sp.latex(f_val)})$\n"
            else:
                loesung += f"- $x = {sp.latex(ns)}$: Sattelpunkt (kein Extremum)\n"
        loesung += "\n"

        # Wendepunkte
        loesung += "**4. Wendepunkte von f** (Extrema von f'):\n"
        for wp in extrema_fp:
            f_val = f_sym.subs(x_sym, wp)
            loesung += f"- $x = {sp.latex(wp)}$: Wendepunkt bei $({sp.latex(wp)} | {sp.latex(f_val)})$\n"

        # Auch f zeigen
        loesung += "\n**Zur Kontrolle — so sieht f aus:**\n"
        display(Markdown(f'<details>\n<summary><b>Lösung aufklappen</b></summary>\n\n{loesung}\n\n</details>'))

        # f-Graph in der Lösung
        out_fgraph = Output()
        with out_fgraph:
            f_num = sp.lambdify(x_sym, f_sym, 'numpy')
            fig2, ax2 = plt.subplots(figsize=(10, 4))
            ax2.plot(xs, f_num(xs), color='#58C4DD', linewidth=2.5, label=rf"$f(x) = {sp.latex(f_sym)}$")
            ax2.plot(xs, fp_vals, '--', color='#FC6255', linewidth=1.5, alpha=0.5, label=rf"$f'(x)$")
            # Extrema markieren
            for ns in nullstellen_fp:
                f_val = float(f_sym.subs(x_sym, ns))
                ax2.plot(float(ns), f_val, 'go', markersize=10, zorder=5)
            # Wendepunkte markieren
            for wp in extrema_fp:
                f_val = float(f_sym.subs(x_sym, wp))
                ax2.plot(float(wp), f_val, 'o', color='orange', markersize=8, zorder=5)
            ax2.axhline(y=0, color='k', linewidth=0.5)
            ax2.grid(True, alpha=0.3)
            ax2.set_xlim(xlim)
            ax2.legend(fontsize=12, loc='upper right', framealpha=0.9)
            ax2.set_title("f(x) mit Extrema (grün) und Wendepunkten (orange)", fontsize=13)
            plt.tight_layout()
            plt.show()
        display(out_fgraph)

btn_spez = Button(description='Neue Aufgabe', button_style='warning', icon='refresh')
btn_spez.on_click(spezial_neu)
display(btn_spez)
display(out_spezial)
spezial_neu()

---
## Zusammenfassung: Die Kernregeln

| Von ... | ... zu ... | Zusammenhang |
|---------|-----------|-------------|
| f' Nullstelle (mit VZW) | f | Extremum von f |
| f' positiv | f | f steigt |
| f' negativ | f | f fällt |
| f' Extremum | f | Wendepunkt von f |
| f'' > 0 | f | linksgekrümmt |
| f'' < 0 | f | rechtsgekrümmt |
| f Nullstelle (mit VZW) | F | Extremum von F |
| f positiv | F | F steigt |
| f negativ | F | F fällt |

**Merksatz:** Eine Stufe hoch = Steigung ablesen. Eine Stufe runter = Fläche (Integral).